In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import xarray as xr
import numpy as np

# 1. Open ALL 20 files at exactly the same time using a wildcard (*)
# The '*' tells Python: "Grab every file in this folder that ends in .nc"
# combine='by_coords' ensures they are glued together in perfect chronological order!
print("Stitching 20 years of data together. This might take a few seconds...")
ds_20yr = xr.open_mfdataset('/content/drive/MyDrive/Colab Notebooks/#11 Project/*.nc', combine='by_coords')

# 2. Crop the massive 20-year dataset to the Mahanadi Basin
# (Lat: 17.0 to 24.5, Lon: 80.0 to 87.5)
cropped_20yr_ds = ds_20yr.sel(LATITUDE=slice(17.0, 24.5), LONGITUDE=slice(80.0, 87.5))

# 3. Extract the final clean NumPy array for the AI
# Remember, the variable name is exactly 'RAINFALL'
cnn_input_20yr = cropped_20yr_ds['RAINFALL'].values

# Let's check the final shape!
print("Success! All years combined.")
print(f"Final 20-Year Tensor Shape: {cnn_input_20yr.shape}")

Stitching 20 years of data together. This might take a few seconds...
Success! All years combined.
Final 20-Year Tensor Shape: (7670, 31, 31)


In [ ]:
!pip install cdsapi
import cdsapi
import os

c = cdsapi.Client(
    url='https://cds.climate.copernicus.eu/api',
    key='91c0ca32-abdf-4b66-8530-25a4f1155b4d' # <--- PASTE YOUR TOKEN HERE
)

print("Starting yearly chunked downloads to bypass server limits...")

# We will loop through the years 2005 to 2025, one at a time
for year in range(2019, 2020):
    file_name = f'ERA5_Mahanadi_{year}.nc'

    # This prevents redownloading if your Colab disconnects midway!
    if os.path.exists(file_name):
        print(f"✅ {year} already downloaded, skipping...")
        continue

    print(f"⏳ Requesting data for {year}...")

    # We ask for just ONE year in this specific request
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': [
                '2m_temperature',
                'mean_sea_level_pressure',
                'specific_humidity',
                '10m_u_component_of_wind',
                '10m_v_component_of_wind'
            ],
            'year': str(year), # <--- Only asking for the current loop year
            'month': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'],
            'day': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12',
                    '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24',
                    '25', '26', '27', '28', '29', '30', '31'],
            'time': ['12:00'],
            'area': [24.5, 80.0, 17.0, 87.5],
            'format': 'netcdf',
        },
        file_name
    )

print("🎉 All 21 years downloaded successfully!")

Starting yearly chunked downloads to bypass server limits...
⏳ Requesting data for 2019...


2026-03-24 14:46:38,143 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

2f02eb182513957fed603e1fe4ea2fda.nc:   0%|          | 0.00/2.75M [00:00<?, ?B/s]

🎉 All 21 years downloaded successfully!


In [ ]:
!pip install cdsapi
import cdsapi
import os

c = cdsapi.Client(
    url='https://cds.climate.copernicus.eu/api',
    key='91c0ca32-abdf-4b66-8530-25a4f1155b4d' # <--- PASTE YOUR TOKEN HERE
)

print("Downloading ONLY the missing Humidity data...")

for year in range(2005, 2026):
    file_name = f'ERA5_Humidity_{year}.nc'

    if os.path.exists(file_name):
        print(f"✅ {year} already downloaded, skipping...")
        continue

    print(f"⏳ Requesting humidity for {year}...")

    c.retrieve(
        'reanalysis-era5-pressure-levels', # <--- The correct cloud-level database!
        {
            'product_type': 'reanalysis',
            'variable': ['specific_humidity'],
            'pressure_level': ['850'], # 850 hPa is the sweet spot for monsoon clouds
            'year': str(year),
            'month': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'],
            'day': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12',
                    '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24',
                    '25', '26', '27', '28', '29', '30', '31'],
            'time': ['12:00'],
            'area': [24.5, 80.0, 17.0, 87.5],
            'format': 'netcdf',
        },
        file_name
    )

print("🎉 Humidity downloaded successfully! We now have all 5 variables.")

⏳ Requesting humidity for 2005...


2026-03-25 14:20:49,254 INFO Request ID is b08a24ef-45ea-4a76-9ed2-c5fb4c2a14ea
INFO:ecmwf.datastores.legacy_client:Request ID is b08a24ef-45ea-4a76-9ed2-c5fb4c2a14ea
2026-03-25 14:20:49,394 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-03-25 14:20:58,087 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-03-25 14:22:05,508 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


83c33c745e06f15bc13e13d1b9d2997c.nc:   0%|          | 0.00/757k [00:00<?, ?B/s]

⏳ Requesting humidity for 2006...


2026-03-25 14:22:07,634 INFO Request ID is fc029a64-54a3-49bc-9133-aa1e4b9780a3
INFO:ecmwf.datastores.legacy_client:Request ID is fc029a64-54a3-49bc-9133-aa1e4b9780a3
2026-03-25 14:22:07,924 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-03-25 14:22:16,684 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-03-25 14:22:58,363 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

7678f9cdba351ae00c8873ae7b6831f0.nc:   0%|          | 0.00/742k [00:00<?, ?B/s]

⏳ Requesting humidity for 2025...


2026-03-25 14:43:51,362 INFO Request ID is 252d9989-0f3d-487c-bb56-8e7ae01d8652
INFO:ecmwf.datastores.legacy_client:Request ID is 252d9989-0f3d-487c-bb56-8e7ae01d8652
2026-03-25 14:43:51,506 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-03-25 14:44:00,210 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-03-25 14:44:41,952 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


9e4f0ca23565ea09aee606a8824d0245.nc:   0%|          | 0.00/755k [00:00<?, ?B/s]

🎉 Humidity downloaded successfully! We now have all 5 variables.


In [ ]:
import xarray as xr
import numpy as np

print("Drive mounted. Starting the ultimate 3-way data merge...")

# ==========================================
# STEP 1: DEFINE ALL THREE FOLDER PATHS
# (Pre-filled based on your screenshot!)
# ==========================================
imd_path = '/content/drive/MyDrive/Colab Notebooks/#11 Project/Rainfall IMD/*.nc'
era5_surface_path = '/content/drive/MyDrive/Colab Notebooks/#11 Project/ERA5 Data/*.nc'
era5_humidity_path = '/content/drive/MyDrive/Colab Notebooks/#11 Project/ERA5 Humidity/*.nc'

# ==========================================
# STEP 2: LOAD & CROP THE IMD TARGET DATA (y)
# ==========================================
print("1. Processing IMD Rainfall...")
ds_imd = xr.open_mfdataset(imd_path, combine='by_coords')
# Crop to Mahanadi Basin
ds_imd_cropped = ds_imd.sel(LATITUDE=slice(17.0, 24.5), LONGITUDE=slice(80.0, 87.5))
y_target = ds_imd_cropped['RAINFALL'].values

# ==========================================
# STEP 3: LOAD ERA5 SURFACE DATA (Temp, Pressure, Wind)
# ==========================================
print("2. Processing ERA5 Surface Features...")
ds_surface = xr.open_mfdataset(era5_surface_path, combine='by_coords')
ds_surface = ds_surface.sortby('latitude').sortby('longitude')

temp = ds_surface['t2m'].values
pressure = ds_surface['msl'].values
u_wind = ds_surface['u10'].values
v_wind = ds_surface['v10'].values

# ==========================================
# STEP 4: LOAD ERA5 CLOUD HUMIDITY DATA
# ==========================================
print("3. Processing ERA5 Cloud Humidity...")
ds_humidity = xr.open_mfdataset(era5_humidity_path, combine='by_coords')
ds_humidity = ds_humidity.sortby('latitude').sortby('longitude')

# We use .squeeze() because the 850hPa pressure level adds an empty dimension.
# Squeeze flattens it so it matches the shape of the surface variables perfectly!
humidity = ds_humidity['q'].squeeze().values

# ==========================================
# STEP 5: THE FINAL 5-CHANNEL SANDWICH (X)
# ==========================================
print("4. Stacking all 5 variables into the final tensor...")
X_features = np.stack([temp, pressure, humidity, u_wind, v_wind], axis=-1)

# ==========================================
# STEP 6: VERIFICATION
# ==========================================
print("\n ULTIMATE DATA ENGINEERING COMPLETE!")
print(f"Target (Rainfall) Shape: {y_target.shape}")
print(f"Features (Weather) Shape: {X_features.shape}")

if y_target.shape[0] == X_features.shape[0] and y_target.shape[1:3] == X_features.shape[1:3]:
    print(" Shapes match perfectly. You are 100% ready to build the CNN-LSTM!")
else:
    print("⚠️ Warning: Shapes do not match. Check your coordinates!")

In [ ]:
import numpy as np
import os

# main project folder path
save_path = '/content/drive/MyDrive/Colab Notebooks/#11 Project/'
master_file = os.path.join(save_path, 'Mahanadi_DeepLearning_Data.npz')

print("Compressing and saving all data into ONE master file...")
print("This might take a minute or two, please wait...")

# np.savez_compressed takes multiple arrays and zips them into one file.
# We are naming the arrays 'X' and 'y' inside the file so they are easy to grab later!
np.savez_compressed(master_file, X=X_features, y=y_target)

print("✅ Success! Your entire 21-year dataset is locked inside: Mahanadi_DeepLearning_Data.npz")
print("You never have to run the NetCDF merge scripts again.")